In [2]:
import numpy as np
import pandas as pd
from scipy.integrate import odeint

# Step 1: Define UAV structural and damage parameters
class UAVDamageSimulator:
    def __init__(self, mass=0.1, stiffness=1000, damping=10):
        self.mass = mass
        self.stiffness = stiffness
        self.damping = damping

    def apply_damage(self, stiffness_reduction):
        """Apply damage by reducing stiffness"""
        self.k_damaged = self.stiffness * (1 - stiffness_reduction)

    def vibration_eq(self, y, t, force_func):
        """Differential equation for a single degree of freedom vibration"""
        x, dx = y
        ddx = (force_func(t) - self.damping * dx - self.k_damaged * x) / self.mass
        return [dx, ddx]

    def simulate_mems_data(self, time, force_func, stiffness_reduction=0):
        self.apply_damage(stiffness_reduction)
        y0 = [0, 0]  # initial displacement and velocity
        sol = odeint(self.vibration_eq, y0, time, args=(force_func,))
        acceleration = np.gradient(sol[:, 1], time)  # approximation of acceleration 2nd derivative
        noise = np.random.normal(0, 0.01, len(acceleration))  # sensor noise
        return acceleration + noise

# Step 2: Define force input simulating external excitation

def force_input(t):
    # Simple harmonic force with added random disturbance
    base_force = 10 * np.sin(2 * np.pi * 5 * t)  # 5 Hz vibration
    disturbance = 0.5 * np.random.normal(0, 1, len(t)) if isinstance(t, np.ndarray) else 0
    return base_force + disturbance

# Step 3: Simulate multi-axis MEMS data

def generate_3axis_mems_data(flight_time, sampling_rate, stiffness_reduction):
    dt = 1 / sampling_rate
    time = np.arange(0, flight_time, dt)

    simulator = UAVDamageSimulator()
    accel_x = simulator.simulate_mems_data(time, force_input, stiffness_reduction)
    accel_y = simulator.simulate_mems_data(time, force_input, stiffness_reduction)
    accel_z = simulator.simulate_mems_data(time, force_input, stiffness_reduction)

    # Simulated gyroscope data (angular rates) simple sinusoidal
    gyro_x = 0.05 * np.sin(2 * np.pi * 0.1 * time) + 0.01 * np.random.normal(0, 1, len(time))
    gyro_y = 0.03 * np.sin(2 * np.pi * 0.2 * time) + 0.01 * np.random.normal(0, 1, len(time))
    gyro_z = 0.04 * np.sin(2 * np.pi * 0.15 * time) + 0.01 * np.random.normal(0, 1, len(time))

    return {
        'time': time,
        'accel_x': accel_x,
        'accel_y': accel_y,
        'accel_z': accel_z,
        'gyro_x': gyro_x,
        'gyro_y': gyro_y,
        'gyro_z': gyro_z
    }

# Step 4: Simulate strain sensor data

def simulate_strain_data(num_points, damage_severity):
    # Simulate strain readings as baseline plus damage-related increment
    baseline_strain = 1e-5 * np.ones(num_points)
    damage_strain = damage_severity * 5e-6 * np.sin(np.linspace(0, np.pi, num_points))
    noise = np.random.normal(0, 1e-7, num_points)
    return baseline_strain + damage_strain + noise

# Step 5: Create dataset with multiple samples

def create_dataset(num_samples, flight_time, sampling_rate):
    dataset = []
    for i in range(num_samples):
        # Random damage severity between 0 and 0.5
        damage_severity = np.random.uniform(0, 0.5)

        # Generate MEMS data
        mems_data = generate_3axis_mems_data(flight_time, sampling_rate, damage_severity)

        # Generate strain data with 5 sensors
        strain_data = simulate_strain_data(5, damage_severity)

        # Label dictionary
        label = {'damage_type': 'simulated', 'damage_severity': damage_severity}

        # Combine data into one dictionary
        sample = {
            'time': mems_data['time'],
            'accel_x': mems_data['accel_x'],
            'accel_y': mems_data['accel_y'],
            'accel_z': mems_data['accel_z'],
            'gyro_x': mems_data['gyro_x'],
            'gyro_y': mems_data['gyro_y'],
            'gyro_z': mems_data['gyro_z'],
            'strain': strain_data,
            'label': label
        }
        dataset.append(sample)
    return dataset

# Step 6: Export dataset to CSV files

def export_sample_to_csv(sample, filename_prefix):
    df = pd.DataFrame({
        'time': sample['time'],
        'accel_x': sample['accel_x'],
        'accel_y': sample['accel_y'],
        'accel_z': sample['accel_z'],
        'gyro_x': sample['gyro_x'],
        'gyro_y': sample['gyro_y'],
        'gyro_z': sample['gyro_z'],
    })
    for i in range(len(sample['strain'])):
        df[f'strain_{i+1}'] = sample['strain'][i]

    # Add label columns
    df['damage_type'] = sample['label']['damage_type']
    df['damage_severity'] = sample['label']['damage_severity']

    df.to_csv(f'{filename_prefix}.csv', index=False)

# Example Usage

num_samples = 5  # Number of flight samples to simulate
flight_time = 10  # flight time in seconds
sampling_rate = 100  # sensor sampling rate in Hz

dataset = create_dataset(num_samples, flight_time, sampling_rate)

# Export the first sample
export_sample_to_csv(dataset[0], 'uav_simulated_sample_1')



In [3]:
import numpy as np
import pandas as pd
from scipy.integrate import odeint

# Step 1: Define UAV structural and damage parameters
class UAVDamageSimulator:
    def __init__(self, mass=0.1, stiffness=1000, damping=10):
        self.mass = mass
        self.stiffness = stiffness
        self.damping = damping

    def apply_damage(self, stiffness_reduction):
        """Apply damage by reducing stiffness"""
        self.k_damaged = self.stiffness * (1 - stiffness_reduction)

    def vibration_eq(self, y, t, force_func):
        """Differential equation for a single degree of freedom vibration"""
        x, dx = y
        ddx = (force_func(t) - self.damping * dx - self.k_damaged * x) / self.mass
        return [dx, ddx]

    def simulate_mems_data(self, time, force_func, stiffness_reduction=0):
        self.apply_damage(stiffness_reduction)
        y0 = [0, 0]  # initial displacement and velocity
        sol = odeint(self.vibration_eq, y0, time, args=(force_func,))
        acceleration = np.gradient(sol[:, 1], time)  # approximation of acceleration 2nd derivative
        noise = np.random.normal(0, 0.01, len(acceleration))  # sensor noise
        return acceleration + noise

# Step 2: Define force input simulating external excitation

def force_input(t):
    # Simple harmonic force with added random disturbance
    base_force = 10 * np.sin(2 * np.pi * 5 * t)  # 5 Hz vibration
    disturbance = 0.5 * np.random.normal(0, 1, len(t)) if isinstance(t, np.ndarray) else 0
    return base_force + disturbance

# Step 3: Simulate multi-axis MEMS data

def generate_3axis_mems_data(flight_time, sampling_rate, stiffness_reduction):
    dt = 1 / sampling_rate
    time = np.arange(0, flight_time, dt)

    simulator = UAVDamageSimulator()
    accel_x = simulator.simulate_mems_data(time, force_input, stiffness_reduction)
    accel_y = simulator.simulate_mems_data(time, force_input, stiffness_reduction)
    accel_z = simulator.simulate_mems_data(time, force_input, stiffness_reduction)

    # Simulated gyroscope data (angular rates) simple sinusoidal
    gyro_x = 0.05 * np.sin(2 * np.pi * 0.1 * time) + 0.01 * np.random.normal(0, 1, len(time))
    gyro_y = 0.03 * np.sin(2 * np.pi * 0.2 * time) + 0.01 * np.random.normal(0, 1, len(time))
    gyro_z = 0.04 * np.sin(2 * np.pi * 0.15 * time) + 0.01 * np.random.normal(0, 1, len(time))

    return {
        'time': time,
        'accel_x': accel_x,
        'accel_y': accel_y,
        'accel_z': accel_z,
        'gyro_x': gyro_x,
        'gyro_y': gyro_y,
        'gyro_z': gyro_z
    }

# Step 4: Simulate strain sensor data

def simulate_strain_data(num_points, damage_severity):
    # Simulate strain readings as baseline plus damage-related increment
    baseline_strain = 1e-5 * np.ones(num_points)
    damage_strain = damage_severity * 5e-6 * np.sin(np.linspace(0, np.pi, num_points))
    noise = np.random.normal(0, 1e-7, num_points)
    return baseline_strain + damage_strain + noise

# Step 5: Create dataset with multiple samples

def create_dataset(num_samples, flight_time, sampling_rate):
    dataset = []
    for i in range(num_samples):
        # Random damage severity between 0 and 0.5
        damage_severity = np.random.uniform(0, 0.5)

        # Generate MEMS data
        mems_data = generate_3axis_mems_data(flight_time, sampling_rate, damage_severity)

        # Generate strain data with 5 sensors
        strain_data = simulate_strain_data(5, damage_severity)

        # Label dictionary
        label = {'damage_type': 'simulated', 'damage_severity': damage_severity}

        # Combine data into one dictionary
        sample = {
            'time': mems_data['time'],
            'accel_x': mems_data['accel_x'],
            'accel_y': mems_data['accel_y'],
            'accel_z': mems_data['accel_z'],
            'gyro_x': mems_data['gyro_x'],
            'gyro_y': mems_data['gyro_y'],
            'gyro_z': mems_data['gyro_z'],
            'strain': strain_data,
            'label': label
        }
        dataset.append(sample)
    return dataset

# Step 6: Export dataset to CSV files

def export_sample_to_csv(sample, filename_prefix):
    df = pd.DataFrame({
        'time': sample['time'],
        'accel_x': sample['accel_x'],
        'accel_y': sample['accel_y'],
        'accel_z': sample['accel_z'],
        'gyro_x': sample['gyro_x'],
        'gyro_y': sample['gyro_y'],
        'gyro_z': sample['gyro_z'],
    })
    for i in range(len(sample['strain'])):
        df[f'strain_{i+1}'] = sample['strain'][i]

    # Add label columns
    df['damage_type'] = sample['label']['damage_type']
    df['damage_severity'] = sample['label']['damage_severity']

    df.to_csv(f'{filename_prefix}.csv', index=False)

# Example Usage

num_samples = 5  # Number of flight samples to simulate
flight_time = 10  # flight time in seconds
sampling_rate = 100  # sensor sampling rate in Hz

dataset = create_dataset(num_samples, flight_time, sampling_rate)

# Export the first sample
export_sample_to_csv(dataset[0], 'uav_simulated_sample_2')
